# Runbook: Data Missing Investigation

## Purpose
Investigate when customer reports missing data in their pipeline.

## When to Use
- Customer reports data not appearing in reports
- Data counts don't match source systems
- Scheduled job completed but no data visible

## Prerequisites
- Customer ID or account identifier
- Date range of missing data
- Source system information

## Incident Information
Fill this out at the start of investigation:
- **Date/Time Reported**: 
- **Reporter**: 
- **Customer ID**: 
- **Affected Date Range**: 
- **Expected Row Count**: 
- **Actual Row Count**: 

In [ ]:
# Setup - Initialize Spark session
import os
import sys
os.environ['SPARK_HOME'] = '/opt/spark'
sys.path.insert(0, '/opt/spark/python')
sys.path.insert(0, '/opt/spark/python/lib/py4j-0.10.9.7-src.zip')

from pyspark.sql import SparkSession
from datetime import datetime, timedelta
import pandas as pd

spark = SparkSession.builder \
    .appName("MissingDataInvestigation") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("✅ Spark session initialized")

In [ ]:
# PARAMETERS - Update these for each investigation
CUSTOMER_ID = "ABC123"  # Change this
START_DATE = "2024-02-01"  # Change this
END_DATE = "2024-02-10"  # Change this
TABLE_NAME = "orders"  # Change this (orders, customers, transactions, etc.)

print(f"Investigating missing data for:")
print(f"  Customer: {CUSTOMER_ID}")
print(f"  Date Range: {START_DATE} to {END_DATE}")
print(f"  Table: {TABLE_NAME}")

## Step 1: Check Raw Layer
First, verify if data arrived from the source system

In [ ]:
# Query raw layer for customer data
raw_query = f"""
SELECT COUNT(*) as row_count,
       MIN(created_date) as earliest_date,
       MAX(created_date) as latest_date
FROM nessie.raw.{TABLE_NAME}
WHERE customer_id = '{CUSTOMER_ID}'
  AND created_date >= '{START_DATE}'
  AND created_date <= '{END_DATE}'
"""

raw_result = spark.sql(raw_query)
print("\n=== RAW LAYER ===")
raw_result.show()

# Sample rows
print("\nSample rows:")
spark.sql(f"SELECT * FROM nessie.raw.{TABLE_NAME} WHERE customer_id = '{CUSTOMER_ID}' LIMIT 5").show()

## Step 2: Check Bronze Layer
Verify data made it through initial ingestion

In [ ]:
# Query bronze layer
bronze_query = f"""
SELECT COUNT(*) as row_count,
       MIN(created_date) as earliest_date,
       MAX(created_date) as latest_date
FROM nessie.bronze.{TABLE_NAME}
WHERE customer_id = '{CUSTOMER_ID}'
  AND created_date >= '{START_DATE}'
  AND created_date <= '{END_DATE}'
"""

bronze_result = spark.sql(bronze_query)
print("\n=== BRONZE LAYER ===")
bronze_result.show()

## Step 3: Check Silver Layer
Verify transformations processed the data

In [ ]:
# Query silver layer
silver_query = f"""
SELECT COUNT(*) as row_count,
       MIN(created_date) as earliest_date,
       MAX(created_date) as latest_date
FROM nessie.silver.{TABLE_NAME}
WHERE customer_id = '{CUSTOMER_ID}'
  AND created_date >= '{START_DATE}'
  AND created_date <= '{END_DATE}'
"""

silver_result = spark.sql(silver_query)
print("\n=== SILVER LAYER ===")
silver_result.show()

## Step 4: Check Gold Layer
Verify final aggregations/marts

In [ ]:
# Query gold layer  
gold_query = f"""
SELECT COUNT(*) as row_count,
       MIN(created_date) as earliest_date,
       MAX(created_date) as latest_date
FROM nessie.gold.{TABLE_NAME}
WHERE customer_id = '{CUSTOMER_ID}'
  AND created_date >= '{START_DATE}'
  AND created_date <= '{END_DATE}'
"""

gold_result = spark.sql(gold_query)
print("\n=== GOLD LAYER ===")
gold_result.show()

## Step 5: Layer Comparison
Compare counts across all layers to identify where data is lost

In [ ]:
# Create summary comparison
raw_count = raw_result.collect()[0]['row_count']
bronze_count = bronze_result.collect()[0]['row_count']
silver_count = silver_result.collect()[0]['row_count']
gold_count = gold_result.collect()[0]['row_count']

comparison = pd.DataFrame([
    {'Layer': 'Raw', 'Row Count': raw_count, 'Loss': 0},
    {'Layer': 'Bronze', 'Row Count': bronze_count, 'Loss': raw_count - bronze_count},
    {'Layer': 'Silver', 'Row Count': silver_count, 'Loss': bronze_count - silver_count},
    {'Layer': 'Gold', 'Row Count': gold_count, 'Loss': silver_count - gold_count},
])

print("\n=== LAYER COMPARISON ===")
print(comparison.to_string(index=False))

# Identify problem layer
if raw_count == 0:
    print("\n⚠️  PROBLEM: No data in RAW layer - source ingestion issue")
elif bronze_count < raw_count:
    print("\n⚠️  PROBLEM: Data lost between RAW → BRONZE")
elif silver_count < bronze_count:
    print("\n⚠️  PROBLEM: Data lost during BRONZE → SILVER transformation")
elif gold_count < silver_count:
    print("\n⚠️  PROBLEM: Data lost during SILVER → GOLD aggregation")
else:
    print("\n✅ Data present in all layers")

## Step 6: Time Travel Analysis
Check if data existed before and was deleted/modified

In [ ]:
# Check table history
print("\n=== TABLE HISTORY ===")
history = spark.sql(f"SELECT * FROM nessie.raw.{TABLE_NAME}.history")
history.show(truncate=False)

# Get snapshots
print("\n=== SNAPSHOTS ===")
snapshots = spark.sql(f"SELECT * FROM nessie.raw.{TABLE_NAME}.snapshots")
snapshots.show(truncate=False)

## Step 7: Check Schema
Verify schema hasn't changed unexpectedly

In [ ]:
# Check current schema
print("\n=== CURRENT SCHEMA ===")
spark.sql(f"DESCRIBE nessie.raw.{TABLE_NAME}").show(truncate=False)

## Resolution

### Findings:
Document what you discovered:
- 

### Root Cause:
- 

### Actions Taken:
1. 
2. 

### Follow-up Required:
- [ ] Monitor customer data for next 24 hours
- [ ] Update documentation if needed
- [ ] Create ticket for permanent fix
- [ ] Notify customer of resolution